In [ ]:
from swiftgalaxy import SWIFTGalaxy, SOAP, MaskCollection, SWIFTGalaxies
from scipy.spatial.transform import Rotation
from swiftsimio.visualisation import project_gas, project_pixel_grid
import matplotlib.patches as patches


import numpy as np, scipy, matplotlib.pyplot as plt, unyt as u, swiftsimio as sw, pandas as pd, glob, re

from matplotlib.lines import Line2D

### Note:

This approach was messy, and thus redefined when looking at the temporal evolution. Halo properties and anisotropies
are now calculated together in the same slurm script. see hpc/L025m5_evo.py

In [ ]:
def halo_properties(sg):
    '''
    Obtain halo properties of candidate galaxy within r200c

    Output:
    t_mass : Total mass 
    gas_frac : gas fraction 
    mmbh_mass : mass of most massive black within
    stellar_corot : kappa_corot of stars within 30kpc
    s_count : star count within 10kpc
    idx : soap index of halo
    '''

    idx = sg.halo_catalogue.soap_index

    g_mass = sg.halo_catalogue.spherical_overdensity_200_crit.gas_mass.to('Msun').value[0]
    t_mass = sg.halo_catalogue.spherical_overdensity_200_crit.total_mass.to('Msun').value[0]

    gas_frac = g_mass / t_mass

    s_count = sg.halo_catalogue.exclusive_sphere_10kpc.number_of_star_particles.value[0]
    mmbh_mass = sg.halo_catalogue.spherical_overdensity_200_crit.most_massive_black_hole_mass.to('Msun').value[0]
    stellar_corot = sg.halo_catalogue.exclusive_sphere_30kpc.kappa_corot_stars.value[0]

    return idx, t_mass, gas_frac, mmbh_mass, stellar_corot, s_count

def halo_anisotropy(sg):
    '''
    Mask gas particles

    Rotate via AM vector to edge on

    Project to 2D image as a histogram
    '''
    img = rotate_and_project_nr(sg)

    return sg.halo_catalogue.soap_index, pixel_anisotropy(img)

## Load L200m6 Box

In [ ]:
catalogue_file_path = # path to L200m6 catalogue file here, z = 0
snapshot_file_path = # path to L200m6 snapshot file here, z = 0 ...

sd = sw.load(catalogue_file_path)

In [ ]:
m200c = sd.spherical_overdensity_200_crit.total_mass


candidates = np.argwhere(
    np.logical_and(
        m200c
        > sw.cosmo_quantity(
            10**(11.5), u.Msun, comoving=True, scale_factor=sd.metadata.a, scale_exponent=0
        ),
        m200c
        < sw.cosmo_quantity(
            1e13, u.Msun, comoving=True, scale_factor=sd.metadata.a, scale_exponent=0
        ),
    )
).squeeze()

#1e11.5 to 1e13


print(f'{len(candidates)} Halos Found in given mass range')
print(f'{len(m200c)} Halos in total')

## Generate Swift Galaxies type for all candidates ready for analysis

In [ ]:
soap = SOAP(catalogue_file_path, soap_index=candidates)
sgs = SWIFTGalaxies(snapshot_file_path, soap)

In [ ]:
raw_data = sgs.map(halo_properties)


In [ ]:
df = pd.DataFrame(raw_data, columns=['halo_index', 'mass', 'gas_frac', 'mmbh', 'corot', 'count'])
df.head()

df.to_csv('halo_properties_L200m6.csv', index=False)

print("Data saved successfully to halo_properties_L200m6.csv")


# Stores all halo properties, anisotropies calculated later. see hpc/L200m6_anisotropies.py